# EXP15: Intraday Adaptive 阈值精细扫描

当前 choppy=0.35, kc_only=0.60。
测试 choppy [0.25-0.50] x kc_only [0.45-0.75] 网格

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
from backtest.stats import print_ranked
import pandas as pd

data = DataBundle.load_default()

BASE_KWARGS = {**C12_KWARGS, "intraday_adaptive": True, "spread_cost": 0.50}
print('Data loaded')

In [ ]:
results = []
for choppy in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    for kc_only in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]:
        if kc_only <= choppy:
            continue
        r = run_variant(data, f"Ch{choppy}/KC{kc_only}",
            **{**BASE_KWARGS, "choppy_threshold": choppy, "kc_only_threshold": kc_only})
        r['choppy'] = choppy
        r['kc_only'] = kc_only
        results.append(r)

print_ranked(results)

In [ ]:
df = pd.DataFrame([{'choppy': r['choppy'], 'kc_only': r['kc_only'],
                     'sharpe': r['sharpe'], 'total_pnl': r['total_pnl'], 'n': r['n']} for r in results])

print("\nSharpe Heatmap (rows=choppy, cols=kc_only):")
print(df.pivot_table(index='choppy', columns='kc_only', values='sharpe').round(2))

print("\nTrade Count:")
print(df.pivot_table(index='choppy', columns='kc_only', values='n'))

In [ ]:
# 冠军 K-Fold
best = sorted(results, key=lambda r: r['sharpe'], reverse=True)[0]
print(f"Champion: choppy={best['choppy']}, kc_only={best['kc_only']}, Sharpe={best['sharpe']:.2f}")

champ_kwargs = {**BASE_KWARGS, "choppy_threshold": best['choppy'], "kc_only_threshold": best['kc_only']}
champ_folds = run_kfold(data, champ_kwargs, label_prefix="Champ_")
base_kwargs = {**BASE_KWARGS, "choppy_threshold": 0.35, "kc_only_threshold": 0.60}
base_folds = run_kfold(data, base_kwargs, label_prefix="Base_")

for b, c in zip(base_folds, champ_folds):
    print(f"{b['fold']}: Base={b['sharpe']:.2f}  Champ={c['sharpe']:.2f}  D={c['sharpe']-b['sharpe']:+.2f}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp15_results.json', 'w') as f:
    json.dump(sanitize_for_json(results), f, indent=2)
print('Saved')